# 10 · Conditionals, branch coverage, and human review

## Goal

Add If/Else branching to the renewal-check workflow — routine renewal vs.
escalation vs. mandatory human review above a spend threshold — and prove
every branch is actually exercised by the eval suite, not just the happy
path.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert (Path("../agents/contract-renewal-desk/workflows/renewal-check.yaml")).exists(), "run 09 first"


## Concept

Branch coverage matters here the same way it matters in normal software
testing, for the same reason: a workflow that's only ever been exercised on
its happy path has an unknown number of live bugs in every branch nobody's
hit yet. The difference from `09` is what belongs in a branch condition —
deterministic, auditable logic (`spend_delta > threshold`) — versus what
doesn't: no reasoning, no "let the model decide," inside a workflow step.
That non-determinism belongs in the agent turn that *calls* the workflow,
never inside it.

**Human-review is a first-class step**, not a workaround: above a spend
threshold, the workflow doesn't guess — it stops and waits for a person.
`evals/golden_cases.json#wf-04-human-review` exists specifically to prove
this step fires rather than silently auto-approving.


## Build


In [ ]:
import yaml
from pathlib import Path
workflow_path = Path("../agents/contract-renewal-desk/workflows/renewal-check.yaml")
workflow = yaml.safe_load(workflow_path.read_text())

workflow["steps"].append({
    "id": "decide",
    "type": "if-else",
    "condition": "@{steps.lookupSpend.output.spendRecord.deltaPct} > 20 or @{steps.lookupPerformance.output.performanceRecord.lateDeliveries} > 1",
    "ifTrue": [
        {
            "id": "checkHighValue",
            "type": "if-else",
            "condition": "@{steps.lookupSpend.output.spendRecord.annualValue} > 2000000",
            "ifTrue": [{"id": "humanReview", "type": "human-review", "output": "reviewDecision"}],
            "ifFalse": [{"id": "escalate", "type": "compose", "input": "escalate", "output": "decision"}],
        },
    ],
    "ifFalse": [{"id": "renew", "type": "compose", "input": "renew", "output": "decision"}],
})
workflow_path.write_text(yaml.dump(workflow, sort_keys=False))
print(workflow_path.read_text())


In [ ]:
from csx.pac import copilot_push
import subprocess
copilot_push(Path("../agents/contract-renewal-desk"))
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Verify

Same harness, same golden set, every notebook.


Every branch, not just one — this is the actual point of the notebook.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

branch_cases = load_golden(tags=["workflow"])
ids_hit = {c["id"] for c in branch_cases}
required = {"wf-01-happy-path-renew", "wf-02-escalate-branch", "wf-04-human-review"}
assert required.issubset(ids_hit), f"missing branch coverage for {required - ids_hit}"

suite = run_suite(client, cases=branch_cases, credit_meter=meter, min_pass_rate=0.75)


## Cost


In [ ]:
meter.report_cost("10", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="branch coverage run — all 4 workflow-tagged cases")


## Teardown


In [ ]:
print("No teardown — branching persists; 11 hardens variable sourcing on top of it.")
